# 7-Eleven NPD Framework: Integrated EDA Master Notebook
본 노트북은 프로젝트 내 여러 폴더에 산재해 있던 핵심 EDA 스크립트와 주피터 노트북의 파편들을 하나의 통합된 흐름으로 정리한 문서입니다.

## 1. 신상품 식별 (NPD Identification)
- POS(B2) 데이터에서 상품별 최초 결제 발생일을 추출하여 신상품군을 확정하고, 출시 후 4주 누적 매출을 집계합니다.

In [ ]:
import polars as pl
import datetime
import os
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 한글 폰트
font_candidates = [f.fname for f in fm.fontManager.ttflist if 'NanumGothic' in f.name or 'Malgun' in f.name or '맑은' in f.name]
if font_candidates:
    matplotlib.rc('font', family=fm.FontProperties(fname=font_candidates[0]).get_name())
matplotlib.rcParams['axes.unicode_minus'] = False

ROOT    = r"C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework"
B2_PATH = os.path.join(ROOT, "data", "processed", "POS 전처리 최종", "pos_data_food_final_상품단위변환전.parquet")
B4_PATH = os.path.join(ROOT, "data", "processed", "B4_ITEM_DV_INFO.parquet")
B5_PATH = os.path.join(ROOT, "data", "processed", "B5_MNM_DATA.parquet")

CLUSTER_COLOR_MAP = {0: '#EF4444', 1: '#10B981', 2: '#FF8C42'}
LEGACY_COLOR      = '#0D1F2D'

def get_true_npd_list(b2_path, b4_path, burn_in_days=14):
    print(f"--- [Step 1.0] 식품 전용 순수 NPD 식별 로직 ---")
    food_categories = ['음료', '과자', '유음료', '미반', '면', '냉장', '맥주', '즉석음료', '빵', '전통주',
                       '아이스크림', '조리빵', '즉석 식품', '건강/기호식품', '가공식품', '양주와인', '디저트',
                       '안주', '신선', '간식', '조미료/건물', '냉동']

    b4_df = pl.read_parquet(b4_path)
    if "ITEM_CD" in b4_df.columns:
        b4_df = b4_df.rename({"ITEM_CD": "상품코드"})
    b4_df   = b4_df.with_columns(pl.col("상품코드").cast(pl.Utf8))
    b4_food = b4_df.filter(pl.col("ITEM_LGDV_NM").is_in(food_categories))
    food_item_list = b4_food.select("상품코드").to_series().to_list()

    b2_lazy = pl.scan_parquet(b2_path).filter(pl.col("상품코드").is_in(food_item_list))
    start_date = b2_lazy.select(pl.col("영업일자").min()).collect().item()
    start_dt   = datetime.datetime.strptime(str(start_date), "%Y%m%d")
    burn_in_threshold = int((start_dt + datetime.timedelta(days=burn_in_days)).strftime("%Y%m%d"))

    legacy_items = b2_lazy.filter(pl.col("영업일자") <= burn_in_threshold).select("상품코드").unique().collect()
    legacy_set   = set(legacy_items["상품코드"].to_list())

    npd_launch_df = (b2_lazy
        .filter(~pl.col("상품코드").is_in(list(legacy_set)))
        .group_by("상품코드")
        .agg(pl.col("영업일자").min().alias("launch_dt"))
        .collect())
    npd_launch_df = npd_launch_df.with_columns([
        pl.col("launch_dt").cast(pl.Utf8).str.to_date("%Y%m%d").alias("launch_date")
    ])
    npd_launch_df = npd_launch_df.with_columns([
        (pl.col("launch_date") + pl.duration(days=30)).alias("end_date")
    ])
    npd_launch_df = npd_launch_df.with_columns([
        pl.col("end_date").dt.strftime("%Y%m%d").cast(pl.Int64).alias("end_dt")
    ])

    b2_npd_lazy = b2_lazy.filter(pl.col("상품코드").is_in(npd_launch_df["상품코드"].to_list()))
    b2_joined   = b2_npd_lazy.join(npd_launch_df.lazy(), on="상품코드", how="inner")

    npd_sales = (b2_joined
        .filter((pl.col("영업일자") >= pl.col("launch_dt")) & (pl.col("영업일자") <= pl.col("end_dt")))
        .group_by("상품코드")
        .agg(pl.col("매출금액").sum().alias("1month_sales"))
        .collect())
    valid_npd_df  = npd_sales.filter(pl.col("1month_sales") > 0)
    valid_npd_set = set(valid_npd_df["상품코드"].to_list())

    return valid_npd_set, legacy_set, food_item_list, b4_food

TRUE_NPD_SET, LEGACY_SET, FOOD_ITEMS, B4_FOOD_LAZY = get_true_npd_list(B2_PATH, B4_PATH)
print(f"식별된 신상품 수: {len(TRUE_NPD_SET):,}")

## 2. 신상품 1개월 성과 집계

In [ ]:
print("신상품 1개월 매출 기준 집계 중...")
b2_lazy  = pl.scan_parquet(B2_PATH)
npd_list = list(TRUE_NPD_SET)

launch_df = (b2_lazy
    .filter(pl.col("상품코드").is_in(npd_list))
    .group_by("상품코드")
    .agg(pl.col("영업일자").min().alias("launch_dt"))
    .collect())
launch_df = launch_df.with_columns([
    pl.col("launch_dt").cast(pl.Utf8).str.to_date("%Y%m%d").alias("launch_date"),
    (pl.col("launch_dt").cast(pl.Utf8).str.to_date("%Y%m%d") + pl.duration(days=30)).alias("end_date")
])
launch_df = launch_df.with_columns([
    pl.col("end_date").dt.strftime("%Y%m%d").cast(pl.Int64).alias("end_dt")
])

filtered_sales = (b2_lazy
    .filter(pl.col("상품코드").is_in(npd_list))
    .join(launch_df.lazy(), on="상품코드", how="inner")
    .filter((pl.col("영업일자") >= pl.col("launch_dt")) & (pl.col("영업일자") <= pl.col("end_dt")))
    .group_by("상품코드")
    .agg(pl.col("매출금액").sum().alias("1month_sales"))
    .collect())

b4_pd  = B4_FOOD_LAZY.select(["상품코드", "ITEM_MDDV_NM", "ITEM_NM"]).collect().to_pandas().drop_duplicates("상품코드")
b4_pd  = b4_pd.rename(columns={"ITEM_MDDV_NM": "중분류명", "ITEM_NM": "상품명"})
merged = pd.merge(filtered_sales.to_pandas(), b4_pd, on="상품코드", how="left")
print(f"집계 완료: {len(merged):,}개 신상품")

## 3. 카테고리 독점도 (Monopoly Score)

In [ ]:
results = []
for cat, group in merged.groupby("중분류명"):
    sorted_group = group.sort_values(by="1month_sales", ascending=False).reset_index(drop=True)
    total_sales  = sorted_group["1month_sales"].sum()
    sorted_group["cum_ratio"] = sorted_group["1month_sales"].cumsum() / total_sales
    top80_items  = sorted_group[sorted_group["cum_ratio"] <= 0.8]
    results.append({
        "중분류명": cat,
        "NPD 총 상품수": len(sorted_group),
        "매출 80% 견인 NPD 수": len(top80_items),
        "카테고리 총 1개월 매출": total_sales
    })

pareto_df = pd.DataFrame(results).sort_values(by="카테고리 총 1개월 매출", ascending=False)
pareto_df.to_excel(os.path.join(ROOT, "eda", "카테고리분석with파레토.xlsx"), index=False)
print(pareto_df.head(10))

## 3-1. 파레토 필터링 (Pareto Category Filter)

In [ ]:
PARETO_XLSX = os.path.join(ROOT, "eda", "카테고리필터링_pareto기준자르기.xlsx")
pareto_filter_df = pd.read_excel(PARETO_XLSX)

# 사용여부 == 'O' 인 중분류만 선택
survived_cats  = set(pareto_filter_df.loc[pareto_filter_df["사용여부"] == "O", "ITEM_MDDV_NM"].tolist())

b4_full = B4_FOOD_LAZY.select(["상품코드", "ITEM_MDDV_NM", "ITEM_NM"]).collect().to_pandas().drop_duplicates("상품코드")
survived_items       = set(b4_full[b4_full["ITEM_MDDV_NM"].isin(survived_cats)]["상품코드"].tolist())
TRUE_NPD_SET_FILTERED = TRUE_NPD_SET.intersection(survived_items)

print(f"파레토 필터 적용 후 분석 대상 NPD: {len(TRUE_NPD_SET_FILTERED):,}개")

## 3-2. KMeans 생애주기 클러스터링

In [ ]:
N_DAYS        = 56
N_CLUSTERS    = 3
MIN_ACTIVE_DAYS = 7

b2_lazy = pl.scan_parquet(B2_PATH)
npd_filtered_list = list(TRUE_NPD_SET_FILTERED)

launch_df2 = (b2_lazy
    .filter(pl.col("상품코드").is_in(npd_filtered_list))
    .group_by("상품코드")
    .agg(pl.col("영업일자").min().alias("launch_dt"))
    .collect())

daily_sales = (b2_lazy
    .filter(pl.col("상품코드").is_in(npd_filtered_list))
    .group_by(["상품코드", "영업일자"])
    .agg(pl.col("매출금액").sum().alias("daily_sales"))
    .collect())

daily_pd   = daily_sales.to_pandas()
launch_pd  = launch_df2.to_pandas()
daily_pd   = daily_pd.merge(launch_pd, on="상품코드", how="inner")
daily_pd["day_offset"] = daily_pd["영업일자"] - daily_pd["launch_dt"]
daily_pd   = daily_pd[(daily_pd["day_offset"] >= 0) & (daily_pd["day_offset"] < N_DAYS)]

pivot = daily_pd.pivot_table(index="상품코드", columns="day_offset", values="daily_sales", aggfunc="sum").fillna(0)
active_days_count = (pivot > 0).sum(axis=1)
pivot_filtered    = pivot[active_days_count >= MIN_ACTIVE_DAYS].copy()

first_day     = pivot_filtered[0].replace(0, np.nan)
pivot_norm    = pivot_filtered.div(first_day, axis=0).fillna(0).clip(upper=10)

X_scaled = StandardScaler().fit_transform(pivot_norm)
kmeans   = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(X_scaled)

pivot_filtered = pivot_filtered.copy()
pivot_filtered["cluster"] = cluster_labels

PRODUCT_LABEL_MAP = {c: f"클러스터 {c}" for c in range(N_CLUSTERS)}

item_cluster_df = pivot_filtered[["cluster"]].reset_index()
item_cluster_df = item_cluster_df.merge(
    b4_full[["상품코드", "ITEM_MDDV_NM", "ITEM_NM"]].rename(columns={"ITEM_MDDV_NM": "중분류명", "ITEM_NM": "상품명"}),
    on="상품코드", how="left"
)
item_cluster_df["클러스터_레이블"] = item_cluster_df["cluster"].map(PRODUCT_LABEL_MAP)
NPD_CLUSTER_DF = item_cluster_df.copy()

print(f"클러스터링 완료: {len(NPD_CLUSTER_DF):,}개 상품")
print(NPD_CLUSTER_DF["cluster"].value_counts().sort_index())

# 클러스터별 평균 궤적 시각화
fig, axes = plt.subplots(1, N_CLUSTERS, figsize=(16, 4), sharey=True)
for c in range(N_CLUSTERS):
    ax    = axes[c]
    c_idx = pivot_filtered[pivot_filtered["cluster"] == c].index
    sub   = pivot_norm.loc[c_idx].drop(columns=[col for col in pivot_norm.columns if col == "cluster"], errors="ignore")
    for _, row in sub.iterrows():
        ax.plot(row.values, color=CLUSTER_COLOR_MAP[c], alpha=0.08, linewidth=0.7)
    ax.plot(sub.mean().values, color=CLUSTER_COLOR_MAP[c], linewidth=2.5, label="평균")
    ax.set_title(f"C{c} (n={len(c_idx)})", fontsize=12)
    ax.set_xlabel("출시 후 경과일")
    ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8)
fig.suptitle("클러스터별 정규화 매출 궤적 (Day 1 = 1.0)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, "eda", "cluster_trajectories.png"), dpi=150, bbox_inches="tight")
plt.show()

## 3-2-A. 클러스터별 절대 매출 분포 (Violin Plot)

In [ ]:
merged_cluster = merged.merge(
    NPD_CLUSTER_DF[["상품코드", "cluster"]].drop_duplicates("상품코드"),
    on="상품코드", how="inner"
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Violin plot
ax = axes[0]
cluster_ids = sorted(NPD_CLUSTER_DF["cluster"].dropna().unique().astype(int))
data_by_cluster = [merged_cluster[merged_cluster["cluster"] == c]["1month_sales"].values for c in cluster_ids]
parts = ax.violinplot(data_by_cluster, positions=cluster_ids, showmedians=True)
for c, pc in zip(cluster_ids, parts["bodies"]):
    pc.set_facecolor(CLUSTER_COLOR_MAP[c])
    pc.set_alpha(0.7)
ax.set_xticks(cluster_ids)
ax.set_xticklabels([f"C{c}" for c in cluster_ids])
ax.set_title("클러스터별 1개월 절대 매출 분포", fontsize=13)
ax.set_ylabel("1개월 매출 (원)")

# Mean / Median bar chart
ax2 = axes[1]
means   = [merged_cluster[merged_cluster["cluster"] == c]["1month_sales"].mean()   for c in cluster_ids]
medians = [merged_cluster[merged_cluster["cluster"] == c]["1month_sales"].median() for c in cluster_ids]
x = np.arange(len(cluster_ids))
ax2.bar(x - 0.2, means,   width=0.35, label="평균",   color=[CLUSTER_COLOR_MAP[c] for c in cluster_ids], alpha=0.85)
ax2.bar(x + 0.2, medians, width=0.35, label="중앙값", color=[CLUSTER_COLOR_MAP[c] for c in cluster_ids], alpha=0.5, edgecolor='black')
ax2.set_xticks(x)
ax2.set_xticklabels([f"C{c}" for c in cluster_ids])
ax2.set_title("클러스터별 평균/중앙값 매출", fontsize=13)
ax2.set_ylabel("1개월 매출 (원)")
ax2.legend()

plt.suptitle("클러스터 궤적 형태 vs 절대 매출 수준 비교", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, "eda", "cluster_sales_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

## 3-2-B. 기획 자산 매몰 진단 (Sunk-Cost Planning Diagnosis)

- **성공**: 전체 신상품 중 1개월 매출 상위 20% (파레토 대칭)
- **실패**: 전체 신상품 중 1개월 매출 하위 20%
- **중간**: 제외 (산점도에서 표시 안 함)
- 색상: 클러스터 색으로 채색, X=실패, O=성공

In [ ]:
import warnings
warnings.filterwarnings("ignore")

FAIL_PCT        = 0.20
SUCCESS_PCT     = 0.80
MIN_POST_LAUNCH = 2
LOOKBACK_YEARS  = 1
KW_MIN_REPEAT   = 2
TOP_CATS        = 15
TOP_KW_CATS     = 12
TOP_KEYWORDS    = 12

b4_mddv = B4_FOOD_LAZY.select(["상품코드", "ITEM_MDDV_NM", "ITEM_NM"]).collect().to_pandas().drop_duplicates("상품코드")
b4_mddv = b4_mddv.rename(columns={"ITEM_MDDV_NM": "중분류명", "ITEM_NM": "상품명"})

# 1개월 매출 재집계
launch_info = (pl.scan_parquet(B2_PATH)
    .filter(pl.col("상품코드").is_in(list(TRUE_NPD_SET_FILTERED)))
    .group_by("상품코드")
    .agg(pl.col("영업일자").min().alias("launch_dt"))
    .collect().to_pandas())
launch_info["end_dt"] = launch_info["launch_dt"] + 30

b2_pd = (pl.scan_parquet(B2_PATH)
    .filter(pl.col("상품코드").is_in(list(TRUE_NPD_SET_FILTERED)))
    .select(["상품코드", "영업일자", "매출금액"])
    .collect().to_pandas())

b2_pd = b2_pd.merge(launch_info, on="상품코드", how="inner")
b2_1m = b2_pd[(b2_pd["영업일자"] >= b2_pd["launch_dt"]) & (b2_pd["영업일자"] <= b2_pd["end_dt"])]
sales_1m = b2_1m.groupby("상품코드")["매출금액"].sum().reset_index(name="1month_sales")

# 클러스터 병합 (inner join — 클러스터 없는 상품 제외)
merged_info = sales_1m.merge(b4_mddv, on="상품코드", how="left")
merged_info = merged_info.merge(
    NPD_CLUSTER_DF[["상품코드", "cluster"]].drop_duplicates("상품코드"),
    on="상품코드", how="inner"
)

# 성공/실패 정의: 전체 글로벌 기준
merged_info["global_rank_pct"] = merged_info["1month_sales"].rank(pct=True)
merged_info["구분"] = "중간"
merged_info.loc[merged_info["global_rank_pct"] >= SUCCESS_PCT, "구분"] = "성공"
merged_info.loc[merged_info["global_rank_pct"] <= FAIL_PCT,    "구분"] = "실패"

# 전체 기획 수 (성공 + 실패만) per 중분류
sf_only = merged_info[merged_info["구분"].isin(["성공", "실패"])].copy()

cat_stats = (sf_only.groupby("중분류명")
    .agg(total=("상품코드", "count"),
         fail_count=("구분", lambda x: (x == "실패").sum()),
         succ_count=("구분", lambda x: (x == "성공").sum()))
    .reset_index())
cat_stats["fail_ratio"] = cat_stats["fail_count"] / cat_stats["total"]
top_cats = cat_stats.nlargest(TOP_CATS, "total")["중분류명"].tolist()

plot_sub = sf_only[sf_only["중분류명"].isin(top_cats)].copy()
cat_order = (cat_stats[cat_stats["중분류명"].isin(top_cats)]
    .sort_values("fail_count", ascending=False)["중분류명"].tolist())

# --- 산점도 (중간 60% 제외, 클러스터 색 적용) ---
cat_to_y = {cat: i for i, cat in enumerate(cat_order)}
rng = np.random.default_rng(42)

fig, ax = plt.subplots(figsize=(14, max(8, len(cat_order) * 0.55)))

cluster_ids = sorted(NPD_CLUSTER_DF["cluster"].dropna().unique().astype(int))

for _, row in plot_sub.iterrows():
    y   = cat_to_y.get(row["중분류명"])
    if y is None or row["구분"] == "중간":
        continue
    is_fail   = row["구분"] == "실패"
    dot_color = CLUSTER_COLOR_MAP.get(int(row["cluster"]), "#94A3B8")
    jitter    = rng.uniform(-0.35, 0.35)
    ax.scatter(
        np.log1p(row["1month_sales"]),
        y + jitter,
        c=dot_color,
        marker="X" if is_fail else "o",
        s=60 if is_fail else 40,
        alpha=0.75,
        edgecolors="black" if is_fail else "none",
        linewidths=0.4,
        zorder=3 if is_fail else 2
    )

ax.set_yticks(range(len(cat_order)))
ax.set_yticklabels(cat_order, fontsize=9)
ax.set_xlabel("log(1개월 매출+1)", fontsize=11)
ax.set_title(f"기획 자산 매몰 진단: 중분류별 성공/실패 분포\n(전체 상위 20%=성공 O, 하위 20%=실패 X, 중간 60% 제외)", fontsize=13)
ax.grid(axis="x", linestyle="--", alpha=0.4)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="gray", markersize=9, label="성공 (상위 20%)"),
    Line2D([0], [0], marker="X", color="w", markerfacecolor="gray", markersize=9,
           markeredgecolor="black", markeredgewidth=0.5, label="실패 (하위 20%)"),
]
for c in cluster_ids:
    legend_elements.append(
        Line2D([0], [0], marker="s", color="w", markerfacecolor=CLUSTER_COLOR_MAP[c],
               markersize=9, label=f"C{c}")
    )
ax.legend(handles=legend_elements, loc="lower right", fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, "eda", "sunk_cost_diagnosis.png"), dpi=150, bbox_inches="tight")
plt.show()

## 4. B5 프로모션 분석

In [ ]:
b5_pd = pl.read_parquet(B5_PATH).to_pandas()
print(f"B5 행사 데이터: {len(b5_pd):,}행")
print(b5_pd.columns.tolist())
print(b5_pd.head(3))

# 상품코드 컬럼 정규화
if "상품코드" not in b5_pd.columns:
    for c in b5_pd.columns:
        if "ITEM" in c.upper() and "CD" in c.upper():
            b5_pd = b5_pd.rename(columns={c: "상품코드"})
            break
b5_pd["상품코드"] = b5_pd["상품코드"].astype(str)

npd_b5 = b5_pd[b5_pd["상품코드"].isin(TRUE_NPD_SET_FILTERED)].copy()
print(f"\n신상품 행사 건수: {len(npd_b5):,}")

# 상품 → 행사 유형 매핑
event_col = [c for c in b5_pd.columns if "행사" in c or "MNM" in c.upper() or "EVENT" in c.upper()][0]
item_to_events = npd_b5.groupby("상품코드")[event_col].apply(set).to_dict()
print(f"행사 컬럼: {event_col}")

## 5. 동반 구매 네트워크 분석

In [ ]:
from itertools import combinations
from collections import defaultdict
from pyvis.network import Network

print("동반 구매 Lift 계산 중 (영수증 단위)...")

b2_receipt = (pl.scan_parquet(B2_PATH)
    .filter(pl.col("상품코드").is_in(list(TRUE_NPD_SET_FILTERED)))
    .select(["거래번호", "상품코드", "영업일자"])
    .collect().to_pandas())

def compute_item_lift(receipt_df, npd_set, min_support=5):
    item_freq  = receipt_df["상품코드"].value_counts().to_dict()
    n_receipts = receipt_df["거래번호"].nunique()
    pair_freq  = defaultdict(int)
    for _, grp in receipt_df.groupby("거래번호")["상품코드"]:
        items = list(set(grp))
        for a, b in combinations(sorted(items), 2):
            pair_freq[(a, b)] += 1
    rows = []
    for (a, b), cnt in pair_freq.items():
        if cnt < min_support:
            continue
        pa   = item_freq[a] / n_receipts
        pb   = item_freq[b] / n_receipts
        pab  = cnt / n_receipts
        lift = pab / (pa * pb) if pa * pb > 0 else 0
        rows.append({"node": a, "node_right": b, "lift": lift, "co_count": cnt,
                     "a_npd": a in npd_set, "b_npd": b in npd_set})
    return pd.DataFrame(rows)

nat_item_pairs = compute_item_lift(b2_receipt, TRUE_NPD_SET_FILTERED)
print(f"자연 구매 페어 수: {len(nat_item_pairs):,}")

# 프로모션 기간 구분 (B5 기반)
if "item_to_events" in dir():
    promo_items   = set(item_to_events.keys())
    promo_receipt = b2_receipt[b2_receipt["상품코드"].isin(promo_items)]
    promo_item_pairs = compute_item_lift(promo_receipt, TRUE_NPD_SET_FILTERED)
    print(f"프로모션 페어 수: {len(promo_item_pairs):,}")
else:
    promo_item_pairs = nat_item_pairs.copy()

def build_pyvis(pairs_pd, title, html_path, min_lift=1.5, node_filter=None):
    sub = pairs_pd[pairs_pd["lift"] >= min_lift].copy()
    if node_filter is not None:
        sub = sub[sub["node"].isin(node_filter) | sub["node_right"].isin(node_filter)]
    if len(sub) == 0:
        print(f"[{title}] 조건에 맞는 엣지 없음")
        return
    net = Network(height="820px", width="100%", bgcolor="white",
                  font_color="#333333", notebook=False, heading="")
    all_nodes = set(sub["node"]) | set(sub["node_right"])
    npd_nodes = set(sub[sub["a_npd"]]["node"]) | set(sub[sub["b_npd"]]["node_right"])
    for n in all_nodes:
        cluster_id = NPD_CLUSTER_DF.set_index("상품코드")["cluster"].to_dict().get(n)
        color = CLUSTER_COLOR_MAP.get(cluster_id, LEGACY_COLOR) if n in npd_nodes else "#CBD5E1"
        size  = 18 if n in npd_nodes else 10
        label = NPD_CLUSTER_DF.set_index("상품코드")["상품명"].to_dict().get(n, n) if n in npd_nodes else n
        net.add_node(n, label=label[:12], color=color, size=size)
    for _, row in sub.iterrows():
        net.add_edge(row["node"], row["node_right"],
                     value=float(row["lift"]),
                     title=f"lift={row['lift']:.2f} co={row['co_count']}")
    net.set_options("""
    {"physics":{"barnesHut":{"gravitationalConstant":-8000,"centralGravity":0.3,"springLength":120}}}
    """)
    html_content = net.generate_html(notebook=False)
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html_content)
    print(f"저장: {html_path}")

OUT_DIR = os.path.join(ROOT, "eda", "중간발표")
os.makedirs(OUT_DIR, exist_ok=True)

build_pyvis(nat_item_pairs,   "자연구매 네트워크",   os.path.join(OUT_DIR, "network_natural.html"))
build_pyvis(promo_item_pairs, "프로모션 네트워크",  os.path.join(OUT_DIR, "network_promo.html"))

## 5-1. NPD 에고 네트워크 (Ego Network)

In [ ]:
TOP_K_PARTNER    = 10
NPD_PER_CLUSTER  = 10
MIN_LIFT         = 1.5
MIN_LIFT_INTER_NPD = 1.0

def ego_pairs(pairs_pd, npd_set=TRUE_NPD_SET_FILTERED):
    """각 NPD를 허브로 하는 에고 네트워크 엣지 추출"""
    # NPD가 포함된 페어만
    sub = pairs_pd[(pairs_pd["a_npd"] | pairs_pd["b_npd"])].copy()

    # NPD 대표 샘플: 클러스터별 상위 NPD_PER_CLUSTER개
    npd_cluster = NPD_CLUSTER_DF[["상품코드", "cluster"]].set_index("상품코드")["cluster"].to_dict()
    npd_in_pairs = (set(sub[sub["a_npd"]]["node"]) | set(sub[sub["b_npd"]]["node_right"]))
    npd_rep = []
    for c in sorted(CLUSTER_COLOR_MAP.keys()):
        cluster_npd = [n for n in npd_in_pairs if npd_cluster.get(n) == c]
        npd_rep.extend(cluster_npd[:NPD_PER_CLUSTER])

    # 각 NPD rep → 상위 K 파트너
    rows = []
    for npd_node in npd_rep:
        partners = sub[
            (sub["node"] == npd_node) | (sub["node_right"] == npd_node)
        ].copy()
        partners["partner"] = partners.apply(
            lambda r: r["node_right"] if r["node"] == npd_node else r["node"], axis=1
        )
        is_inter_npd = partners["partner"].isin(npd_set)
        lift_thresh  = partners.apply(
            lambda r: MIN_LIFT_INTER_NPD if r["partner"] in npd_set else MIN_LIFT, axis=1
        )
        partners = partners[partners["lift"] >= lift_thresh]
        top_k    = partners.nlargest(TOP_K_PARTNER, "lift")
        for _, r in top_k.iterrows():
            rows.append({"node": npd_node, "node_right": r["partner"],
                         "lift": r["lift"], "co_count": r["co_count"],
                         "a_npd": True, "b_npd": r["partner"] in npd_set})

    return pd.DataFrame(rows).drop(columns=["npd_node"], errors="ignore").reset_index(drop=True)

nat_ego   = ego_pairs(nat_item_pairs)
promo_ego = ego_pairs(promo_item_pairs)
ego_nodes = set(nat_ego["node"]) | set(nat_ego["node_right"]) | set(promo_ego["node"]) | set(promo_ego["node_right"])

build_pyvis(nat_ego,   "NPD 에고 (자연)",   os.path.join(OUT_DIR, "ego_natural.html"),  min_lift=MIN_LIFT_INTER_NPD, node_filter=ego_nodes)
build_pyvis(promo_ego, "NPD 에고 (프로모)", os.path.join(OUT_DIR, "ego_promo.html"),    min_lift=MIN_LIFT_INTER_NPD, node_filter=ego_nodes)

## 6. 중간발표 데이터 추출

In [ ]:
print("중간발표용 엑셀 파일 추출 중...")

# 클러스터별 상품 목록
NPD_CLUSTER_DF.to_excel(os.path.join(OUT_DIR, "npd_cluster_assignment.xlsx"), index=False)

# 성공/실패 레이블 포함 merged_info
merged_info.to_excel(os.path.join(OUT_DIR, "npd_success_failure_labels.xlsx"), index=False)

# 카테고리별 파레토 요약
pareto_df.to_excel(os.path.join(OUT_DIR, "category_pareto_summary.xlsx"), index=False)

print(f"\n저장 완료: {OUT_DIR}")
print("  - npd_cluster_assignment.xlsx")
print("  - npd_success_failure_labels.xlsx")
print("  - category_pareto_summary.xlsx")
print("  - network_natural.html")
print("  - network_promo.html")
print("  - ego_natural.html")
print("  - ego_promo.html")